# Liberaries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, zero_one_loss

# Read Data

In [ ]:
data_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
data_train

In [ ]:
label_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)[0]
label_train

In [ ]:
X_test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
X_test

# Data Properties

In [ ]:
#Training Data Columns
data_train.columns

In [ ]:
#Training Data Description for Quantitative Columns
data_train.describe()

In [ ]:
data_train.info()

**No Missing Values in data_train**

In [ ]:
label_train.info()

**No Missing Values in label_train**

In [ ]:
X_test.info()

**No Missing Values in X_test**

# Numerical & Non_numerical Columns

In [ ]:
numerical = data_train.select_dtypes(include=['number']).columns
numerical

In [ ]:
non_numerical = data_train.select_dtypes(exclude=['number']).columns
non_numerical

**No Non_numerical Columns**

# Quantitative & Categorical Columns

In [ ]:
quantitative, categorical = [], []
for i in data_train.columns:
    if len(data_train[i].unique()) > 10:
        quantitative.append(i)
        
    else:
        categorical.append(i)

In [ ]:
quantitative

In [ ]:
categorical

**No Categorical Columns**

# Data Columns Description 

In [ ]:
for i in data_train.columns:
    print((str(i)+':').center(30))
    print(data_train[i].describe())

# Training Data Ploting

In [ ]:
for i in data_train.columns:
    sns.histplot(data_train[i], kde=True, alpha=0.2, label=i)
    plt.legend()
    plt.show()
    print('='*100)

**From training data description and ploting, We can conclude that all the columns are standard_normal**

# Training Labels Ploting

In [ ]:
#Bar Chart Ploting
a = label_train.unique()
b = label_train.value_counts()
plt.bar(a, b, label='Training Labels', edgecolor='k', width=0.1)
for i, j in zip(a, b):
    plt.text(i, j, j, size=10, ha='center', va='bottom')
plt.legend(loc='upper center')
plt.show()

In [ ]:
#Pie Chart Ploting
plt.pie(b, labels=a, textprops = {'size' : 15} , autopct ='%1.1f%%',
            explode=[0.04 for  i in a] , shadow = True)
plt.show()

# Correlation Matrix

In [ ]:
train_combine = data_train.copy()
train_combine['labels'] = label_train
train_combine

In [ ]:
corr_mat = train_combine.corr().sort_values(by='labels', ascending=False)

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(corr_mat, annot=True, fmt='.2f', annot_kws={'size':6})

In [ ]:
#Positive Correlatoin btween columns[12,28]
sns.scatterplot(x=train_combine[12], y=train_combine[28], hue=train_combine['labels'])

In [ ]:
#Negative Correlatoin btween columns[4,28]
sns.scatterplot(x=train_combine[4], y=train_combine[28], hue=train_combine['labels'])

# Training & Cross_validation Data split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_train, label_train, test_size=0.2,
                                                shuffle=True, random_state=44)
X_val

# Find Best Values of Hyper Parameters 

In [ ]:
logReg = LogisticRegression()
dictionary = {'C':range(1,20,1)}
grid = GridSearchCV(logReg, dictionary)
grid.fit(X_train, y_train)
grid.best_estimator_

In [ ]:
DT = DecisionTreeClassifier()
dictionary = {'criterion':['entropy', 'log_loss','gini'], 'max_depth':range(2,20,1),
              'max_features':range(10,40,5), 'min_samples_split':range(2,20,1)}
randomized = RandomizedSearchCV(DT, dictionary, n_iter=50)
randomized.fit(X_train, y_train)
randomized.best_estimator_

In [ ]:
RF = RandomForestClassifier()
dictionary = {'n_estimators':range(10,110,10), 'criterion':['entropy', 'log_loss','gini'],
              'max_depth':range(2,20,1), 'max_features':range(10,40,5), 'min_samples_split':range(2,20,1)}
randomized = RandomizedSearchCV(RF, dictionary, n_iter=50)
randomized.fit(X_train, y_train)
randomized.best_estimator_

In [ ]:
NB = BernoulliNB()
dictionary = {'alpha':[0.00001,0.00005,0.0001,0.0005,0.001,0.05,0.01,0.05,0.1,0.5]}
grid = GridSearchCV(NB, dictionary)
grid.fit(X_train, y_train)
grid.best_estimator_

In [ ]:
svc = SVC(gamma='scale')
dictionary = {'kernel':['sigmoid', 'rbf', 'poly'], 'C':[0.5,1,2,3], 'degree':[2,3,4,5]}
grid = GridSearchCV(svc, dictionary)
grid.fit(X_train, y_train)
grid.best_estimator_

In [ ]:
KNN = KNeighborsClassifier()
dictionary = {'n_neighbors':range(1,12,2), 'weights':['uniform', 'distance']}
grid = GridSearchCV(KNN, dictionary)
grid.fit(X_train, y_train)
grid.best_estimator_

# Models Definition

In [ ]:
logReg = LogisticRegression(C=4)
DT = DecisionTreeClassifier(criterion='log_loss', max_depth=17, max_features=30,
                       min_samples_split=3)
RF = RandomForestClassifier(criterion='log_loss', max_depth=16, max_features=20,
                       min_samples_split=9)
NB = BernoulliNB(alpha=1e-05)
svc = SVC(C=2, degree=2)
KNN = KNeighborsClassifier()
VC = VotingClassifier(estimators=[('LogisticRegression', logReg), ('DecisionTreeClassifier', DT),
          ('RandomForestClassifier', RF),('BernoulliNB' , NB), ('SVC' , svc), ('KNeighborsClassifier',KNN)])

In [ ]:
Models = {'LogisticRegression':logReg,
          'DecisionTreeClassifier':DT,
          'RandomForestClassifier': RF,
          'BernoulliNB' : NB,
          'SVC' : svc,
          'KNeighborsClassifier' : KNN,
          'VotingClassifier' : VC
         }


# Models Fitting & Evaluating

In [ ]:
best_model = None
best_score = 0.0
for model in Models.keys(): 
    #Model Fitting
    Models[model].fit(X_train, y_train)
    print(str(model).center(30))
    
    #_______________________________________________________
    #Prediction
    y_pred = Models[model].predict(X_val)
    
    #_______________________________________________________
    #metrices
    CM = confusion_matrix(y_val, y_pred)
    F1 = f1_score(y_val, y_pred)
    accuracy = accuracy_score(y_val, y_pred, normalize=False)
    loss = zero_one_loss(y_val, y_pred, normalize=False)
    #_______________________________________________________
    #Score
    print('Training Score:', Models[model].score(X_train, y_train))
    print('Testing Score:', Models[model].score(X_val, y_val))
    
    #_______________________________________________________
    print()
    if F1 > best_score:
        best_score = F1
        best_model = model
    print('confusion_matrix\n',CM)
    print()
    print('f1_score',F1)
    print('Number of right predictions',accuracy)
    print('Number of wrong predictions',loss)
    print('==========================================================')


In [ ]:
print('Best Model:', best_model)
print('Best Score:', best_score)

# Feature Selection

In [ ]:
#Model
LR = LogisticRegression()

In [ ]:
selector = SelectFromModel(LR)
selector.fit(data_train, label_train)

In [ ]:
new_data_train = selector.transform(data_train)
new_data_train =  pd.DataFrame(new_data_train)
new_data_train

In [ ]:
new_X_train, new_X_val, new_y_train, new_y_val = train_test_split(new_data_train, label_train, test_size=0.2,
                                                shuffle=True, random_state=44)
new_X_train

# New Models Fitting & Evaluating

In [ ]:
best_model = None
best_score = 0.0
for model in Models.keys(): 
    #Model Fitting
    Models[model].fit(new_X_train, new_y_train)
    print(str(model).center(30))
    
    #_______________________________________________________
    #Prediction
    new_y_pred = Models[model].predict(new_X_val)
    
    #_______________________________________________________
    #metrices
    CM = confusion_matrix(new_y_val, new_y_pred)
    F1 = f1_score(new_y_val, new_y_pred)
    accuracy = accuracy_score(new_y_val, new_y_pred, normalize=False)
    loss = zero_one_loss(new_y_val, new_y_pred, normalize=False)
    #_______________________________________________________
    #Score
    print('Training Score:', Models[model].score(new_X_train, new_y_train))
    print('Testing Score:', Models[model].score(new_X_val, new_y_val))
    
    #_______________________________________________________
    print()
    if F1 > best_score:
        best_score = F1
        best_model = model
    print('confusion_matrix\n',CM)
    print()
    print('f1_score',F1)
    print('Number of right predictions',accuracy)
    print('Number of wrong predictions',loss)
    print('==========================================================')


In [ ]:
print('Best Model:', best_model)
print('Best Score:', best_score)

**Smaller score after feature selection**

# Submission

In [ ]:
KNN.fit(X_train, y_train)
submission = pd.DataFrame(KNN.predict(X_test), columns=['Solution'])
submission['Id'] = np.arange(len(submission))
submission = submission[['Id', 'Solution']]
submission

In [ ]:
submission.to_csv("submission.csv")